# Experiment: Colab A100 - 32B v3-quick (빠른 테스트)

## v2 → v3 변경점 요약

| # | 변경 항목 | v2 | v3 | 이유 |
|---|----------|----|----|------|
| 1 | **양자화** | 4bit NF4 | **8bit** | Qwen3 4bit 시 MMLU 5pt+ 하락 연구 근거. 96GB면 8bit 충분 |
| 2 | **LoRA rank** | r=64, α=128 | **r=32, α=64** | 데이터 수천 개 규모에서 r=64는 과적합 위험. α/r=2 비율 유지 |
| 3 | **LoRA dropout** | 0.1 | **0.05** | r 낮추면서 dropout도 조정 |
| 4 | **이미지 해상도** | 고정 896×896 | **동적 512²~896²** | aspect ratio 유지 → 라벨 왜곡 방지 |
| 5 | **GradScaler** | 사용 | **제거** | bfloat16은 dynamic range가 fp32와 동일하여 scaler 불필요 |
| 6 | **Gradient Clipping** | 없음 | **max_norm=1.0** | 대형 모델 학습 안정성 |
| 7 | **Scheduler** | linear + warmup 5% | **cosine + warmup 10%** | VLM fine-tuning에서 cosine이 더 안정적 수렴 |
| 8 | **Early Stopping** | 없음 | **patience=2** | 불필요한 epoch 방지 |
| 9 | **시스템 프롬프트** | 기본 | **/no_think 추가** | Qwen3 thinking mode 비활성화 → 불필요한 토큰 생성 방지 |
| 10 | **Inference** | max_new_tokens=4 | **max_new_tokens=2** | 단일 토큰 답변에 2면 충분, 속도 향상 |

## VRAM 추정 (A100 96GB)

| 항목 | 추정 |
|------|------|
| 모델 weights (8bit) | ~32GB |
| LoRA adapter (bf16) | ~0.8GB |
| Optimizer (AdamW m,v) | ~1.6GB |
| Gradients | ~0.8GB |
| Activations (batch=1, ~896px, grad ckpt) | ~15-25GB |
| KV cache / 버퍼 | ~5-10GB |
| **합계** | **~55-70GB** |
| **여유** | **~26-41GB** |

# 환경 준비
- 런타임 -> 런타임 유형 변경 -> GPU (A100)
- 아래 셀 실행 후 런타임 재시작

In [ ]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets pillow pandas --upgrade

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name())
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

# 데이터 준비
구글 드라이브에 데이터 zip 파일을 업로드한 후 실행

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 압축 해제 (경로는 본인 드라이브에 맞게 수정)
!unzip "/content/drive/My Drive/2026-ssafy-15-2-ai.zip" -d "/content/"

# 라이브러리, 데이터, 설정

In [ ]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from collections import Counter
from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup   # [v3] linear → cosine
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ===== 실험 설정 =====
MODEL_ID = "Qwen/Qwen3-VL-32B-Instruct"
MAX_NEW_TOKENS = 2                            # [v3] 8 → 2 (단일 토큰 답변에 충분)
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ===== 데이터 로드 =====
train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")

# ===== dev.csv pseudo-label 추가 =====
dev_df = pd.read_csv("/content/dev.csv")
pseudo_labels = []
for _, row in dev_df.iterrows():
    responses = []
    for i in range(1, 6):
        col = f"응답{i}"
        if col in dev_df.columns and pd.notna(row[col]):
            r = str(row[col]).strip().lower()
            if r in ["a","b","c","d"]:
                responses.append(r)
    if responses:
        pseudo_labels.append(Counter(responses).most_common(1)[0][0])
    else:
        pseudo_labels.append("a")

dev_df["answer"] = pseudo_labels
dev_for_train = dev_df[["id","path","question","a","b","c","d","answer"]].copy()

train_df = pd.concat([train_df, dev_for_train], ignore_index=True)

# [v3-quick] 전체 데이터의 10%만 샘플링 (빠른 테스트용)
train_df = train_df.sample(frac=0.1, random_state=SEED).reset_index(drop=True)
print(f"총 학습 데이터: {len(train_df)}개 (전체의 10% 샘플링)")
print(f"테스트 데이터: {len(test_df)}개")

# 모델, Processor

32B 모델 다운로드: 약 32GB, 15~25분 소요

In [ ]:
# [v3] 4bit NF4 → 8bit 양자화
# Qwen3는 4bit에서 MMLU 5pt+ 하락 (arXiv:2505.02214)
# A100 96GB면 8bit(~32GB)로도 VRAM 여유 충분
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

# [v3] 이미지 해상도: 고정 896² → 동적 범위 512²~896²
# aspect ratio를 유지하면서 재활용품 라벨 왜곡 방지
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=512*512,       # 하한: 작은 이미지도 수용
    max_pixels=896*896,       # 상한: VRAM 안전 범위 내 최대
    trust_remote_code=True,
)

base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# [v3] LoRA: r=64 → r=32, alpha=128 → 64 (α/r=2 유지)
# 데이터 수천 개 규모에서 r=64는 과적합 위험
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,    # [v3] 0.1 → 0.05 (r 낮추면서 조정)
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

# VRAM 사용량 확인
print(f"\nVRAM allocated: {torch.cuda.memory_allocated()/1024**3:.1f} GB")
print(f"VRAM reserved:  {torch.cuda.memory_reserved()/1024**3:.1f} GB")

# 프롬프트 (한국어 재활용 특화)

**왜 한국어?** 재활용품 이미지의 라벨이 한국어이고, 질문도 한국어. 도메인 특화 지시가 범용 영어보다 유리.

**왜 /no_think?** Qwen3-VL은 기본적으로 thinking mode가 켜져 있어 `<think>...</think>` 블록을 생성함. MCQ 단답에는 불필요하므로 비활성화.

In [ ]:
# [v3] /no_think 추가 — Qwen3 thinking mode 비활성화
SYSTEM_INSTRUCT = (
    "/no_think\n"
    "당신은 재활용품 이미지를 분석하는 전문 AI입니다. "
    "이미지의 소재, 색상, 형태, 라벨을 주의 깊게 관찰하세요. "
    "질문에 대해 a, b, c, d 중 정답 하나만 소문자로 출력하세요."
)

def build_mc_prompt(question, a, b, c, d):
    return (
        "이미지를 자세히 살펴보고 아래 질문에 답하세요.\n\n"
        f"질문: {question}\n\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답: "
    )

# Dataset, Collator (Labels 마스킹 적용)

**Labels 마스킹이란?** 시스템/유저 프롬프트는 loss에서 제외 → assistant 답변만 학습해서 효율 극대화.

In [ ]:
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]
            text = self.processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            texts.append(text)
            images.append(img)

        enc = self.processor(
            text=texts, images=images, padding=True, return_tensors="pt"
        )

        if self.train:
            labels = enc["input_ids"].clone()
            for i, text in enumerate(texts):
                assistant_marker = "assistant\n"
                marker_pos = text.rfind(assistant_marker)
                if marker_pos != -1:
                    prefix = text[:marker_pos + len(assistant_marker)]
                    prefix_ids = self.processor.tokenizer(
                        prefix, return_tensors="pt"
                    )["input_ids"].shape[1]
                    labels[i, :prefix_ids] = -100
            enc["labels"] = labels

        return enc

# DataLoader

In [ ]:
split = int(len(train_df)*0.9)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                          collate_fn=DataCollator(processor, True), num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False,
                          collate_fn=DataCollator(processor, True), num_workers=0)

print(f"Train: {len(train_ds)}, Valid: {len(valid_ds)}")

# Fine-tuning (빠른 테스트)

- 1 epoch, 전체 데이터 10%만 사용
- 나머지 설정은 v3와 동일
- A100에서 약 10~20분 예상

In [ ]:
model = model.to(device)
GRAD_ACCUM = 16
EPOCHS = 1              # [v3-quick] 1 epoch (빠른 테스트)
LR = 2e-5
PATIENCE = 99             # [v3-quick] early stopping 사실상 비활성화

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
num_training_steps = EPOCHS * math.ceil(len(train_loader)/GRAD_ACCUM)

# [v3] cosine scheduler + warmup 10% (linear + 5%에서 변경)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(num_training_steps * 0.10),
    num_training_steps=num_training_steps
)
# [v3] GradScaler 제거 — bfloat16은 dynamic range가 fp32와 동일하여 loss scaling 불필요

best_val_loss = float('inf')
global_step = 0
no_improve_count = 0      # [v3] early stopping counter

for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]", unit="batch")
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        loss.backward()                       # [v3] scaler.scale() 제거, 직접 backward
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            # [v3] gradient clipping 추가
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()                  # [v3] scaler.step() → optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1
            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.3f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
            running = 0.0

    # Validation
    model.eval()
    val_loss, val_steps = 0.0, 0
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [valid]", unit="batch"):
            vb = {k:v.to(device) for k,v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    avg_val = val_loss/val_steps
    print(f"[Epoch {epoch+1}] valid loss {avg_val:.4f}")

    # Best model 저장 + Early Stopping
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        no_improve_count = 0
        SAVE_DIR = "/content/drive/My Drive/best_model_colab_a100_v3_quick"
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        print(f"  -> Best model saved! (val_loss: {avg_val:.4f})")
    else:
        no_improve_count += 1
        print(f"  -> No improvement ({no_improve_count}/{PATIENCE})")
        if no_improve_count >= PATIENCE:
            print("Early stopping triggered!")
            break

print(f"\nTraining done. Best valid loss: {best_val_loss:.4f}")
print(f"VRAM peak: {torch.cuda.max_memory_allocated()/1024**3:.1f} GB")

# Inference

전체 test 데이터 추론. 5074개 기준 약 30분~1시간 소요.

In [ ]:
def extract_choice(text: str) -> str:
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last
    tokens = last.split()
    for tok in tokens:
        if tok in ["a", "b", "c", "d"]:
            return tok
    return "a"

model.eval()
preds = []

for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[
            {"type":"image","image":img},
            {"type":"text","text":user_text}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,    # [v3] 4 → 2
            do_sample=False,
            eos_token_id=processor.tokenizer.eos_token_id
        )
    output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    preds.append(extract_choice(output_text))

# 제출 파일 생성 (Google Drive에도 백업)
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("/content/submission_colab_a100_v3_quick.csv", index=False)
submission.to_csv("/content/drive/My Drive/submission_colab_a100_v3_quick.csv", index=False)
print("Saved submission_colab_a100_v3_quick.csv")
print(submission.head(10))